In [ ]:
import json, torch
print(json.dumps({"cuda_available": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "torch": torch.__version__}))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pathlib, zipfile, shutil, subprocess, sys
BASE=pathlib.Path('/content/drive/MyDrive/SP_Lense_GPU_Magnitude_Tuning')
BASE.mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/SP_Lense_tuning_payload.zip', BASE/'SP_Lense_tuning_payload.zip')
PAYLOAD=pathlib.Path('/content/sp_lense_tuning')
with zipfile.ZipFile('/content/SP_Lense_tuning_payload.zip') as z: z.extractall(PAYLOAD)
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==5.15.1'])
print('Minimal tuning bundle saved to Drive; dependencies installed.')

In [ ]:
import importlib.util, json, time
from IPython.display import clear_output
spec=importlib.util.spec_from_file_location('sp_gpu_sweep',PAYLOAD/'gpu_sweep.py')
sweep=importlib.util.module_from_spec(spec); spec.loader.exec_module(sweep)
OUT=BASE/'run_v1'
def dashboard_progress(p):
    clear_output(wait=True)
    print('SP_LENSE_PROGRESS '+json.dumps(p), flush=True)
try:
    result=sweep.main(PAYLOAD,OUT,callback=dashboard_progress)
    print('SP_LENSE_RESULT '+json.dumps(result),flush=True)
except BaseException as e:
    failure={'stage':'failed','completed':0,'total':1,'state':'failed','error':repr(e)}
    (OUT/'FAILURE.json').write_text(json.dumps(failure))
    dashboard_progress(failure)
    raise

In [ ]:
import shutil, importlib.util, sys, gc, torch, json
for name in ('last_traceback','last_value','last_exc'):
    if hasattr(sys,name): setattr(sys,name,None)
gc.collect(); torch.cuda.empty_cache()
shutil.copy2('/content/gpu_sweep.py', BASE/'gpu_sweep_v2.py')
spec=importlib.util.spec_from_file_location('sp_gpu_sweep_v2',BASE/'gpu_sweep_v2.py')
sweep2=importlib.util.module_from_spec(spec);spec.loader.exec_module(sweep2)
OUT2=BASE/'run_v2'
try:
    result2=sweep2.main(PAYLOAD,OUT2,callback=dashboard_progress)
    print('SP_LENSE_RESULT '+json.dumps(result2),flush=True)
except BaseException as e:
    failure={'stage':'failed','completed':0,'total':1,'state':'failed','error':repr(e)}
    (OUT2/'FAILURE.json').write_text(json.dumps(failure))
    dashboard_progress(failure)
    raise